# Web后端开发 06：AI聊天后端项目

上一课你完成了能读写SQLite文件的repository，但数据访问层还没有接到HTTP上；第04课的分层工程保存在内存里，程序一退出数据就消失。本课把“FastAPI分层工程结构 + Pydantic数据模型 + SQLite数据访问层”组合成一个真正可运行、可测试、可持久化的**AI聊天后端**。

> 这是一节综合项目课。Notebook会先带你复习和补齐需要的概念，再逐步演示每一层怎么实现，最后由你独立完成整个`chat_app`工程。示例业务使用“学习卡片”，作业业务使用“AI聊天”，两者结构相同但名称不同，避免直接复制答案。


## 1. 本课目标与范围

完成本课后，你应该能够：

1. 用自己的话解释会话、消息、角色、外键这些聊天后端的基本概念；
2. 用`conversations`和`messages`两张表保存聊天数据，并用外键表达“消息属于某个会话”；
3. 把repository接入FastAPI分层工程，让数据真正写入SQLite文件；
4. 在services层实现“发送消息”的完整业务流程：检查会话、保存用户消息、生成AI回复、保存回复、返回回复；
5. 把“AI回复”抽象成独立的`ai_provider`函数，理解为什么测试使用模拟回复而不是真实大模型API；
6. 用路由层处理请求并把“会话不存在”转换成HTTP 404响应；
7. 用`lifespan`在应用启动时初始化数据库；
8. 运行一套完整的pytest测试验收整个API。

本课**不**学习：真实大模型API的调用（只讲概念）、用户登录鉴权、流式输出、消息加密、数据库迁移工具。这些需要在后续课程中展开。


## 2. 运行本课前：环境和依赖

本课需要`fastapi`、`pydantic`、`uvicorn`、`httpx2`和`pytest`。请先在PowerShell中进入本课目录并执行：

```powershell
cd "02_Web后端开发/06_AI聊天后端项目"
python -m pip install -r requirements.txt
```

- `python`：当前环境的Python解释器；
- `-m pip`：让当前Python运行自己的pip模块；
- `install`：安装依赖；
- `-r requirements.txt`：从依赖文件读取要安装的包。

也可以运行下一格的`%pip`，它会向当前Notebook内核安装依赖；安装后若导入仍失败，请重启内核。

`requirements.txt`中每个包的用途：

- `fastapi`：Web框架，提供路由、请求解析、自动文档；
- `pydantic`：数据校验库，`fastapi`依赖它解析请求体和响应模型；
- `uvicorn`：ASGI服务器，用于把应用真正跑起来（本课作业用`uvicorn chat_app.main:app`启动）；
- `httpx2`：HTTP客户端库，`TestClient`依赖它模拟请求（`TestClient`是FastAPI提供的测试客户端，不是独立服务器；新版`starlette`的测试客户端优先使用`httpx2`，不再依赖`httpx`）；
- `pytest`：测试框架，用于运行课末作业测试。

`sqlite3`是Python标准库，不需要也不能用pip安装。


In [ ]:
# 如果已在PowerShell安装，可以跳过这一格。
%pip install -r requirements.txt


下面首次导入本课使用的第三方库，并打印版本，确认安装成功。

- `fastapi`：Web框架；
- `pydantic`：数据校验库；
- `httpx2`：HTTP客户端库，测试客户端依赖它；
- `pytest`：测试框架；
- `uvicorn`：ASGI服务器。

每个库的`__version__`是安装版本的字符串，只用于确认依赖可用。


In [2]:
import fastapi
import httpx2
import pydantic
import pytest
import uvicorn

print("fastapi：", fastapi.__version__)
print("pydantic：", pydantic.__version__)
print("httpx2：", httpx2.__version__)
print("pytest：", pytest.__version__)
print("uvicorn：", uvicorn.__version__)


fastapi： 0.141.1
pydantic： 2.13.4
httpx2： 2.9.1
pytest： 8.4.2
uvicorn： 0.40.0


## 3. 整体流程总览

在动手写代码前，先看一次“用户发送消息”请求的完整链路，本课后面会逐层实现：

```text
客户端（浏览器/测试）
   │  POST /conversations/1/messages
   │  {"content": "外键是什么"}
   ▼
FastAPI路由层 routers/chat.py
   │  解析路径参数、校验请求体 → 调用services
   ▼
业务逻辑层 services.py
   │  检查会话存在 → 保存用户消息 → 生成回复 → 保存回复
   ▼
数据访问层 repository.py          AI回复生成 ai_provider.py
   │  写SQL、管连接和事务             │  规则回复（本课）/真实API（未来）
   ▼                                │
SQLite数据库 conversations/messages  ┘
   │
   ▼
FastAPI返回201响应：AI回复消息JSON
```

其中：

- 请求和JSON校验、路由分发、响应序列化由**FastAPI框架**自动完成；
- 路由函数、services、repository、ai_provider是我们自己写的**用户代码**；
- 数据库文件读写由**SQLite**完成，Python用标准库`sqlite3`连接它。

学习本课时，请时刻问自己：这一步是框架做的，还是我写的代码做的，还是数据库做的？


## 4. 业务对象：会话、消息、角色

一个聊天后端最少要管理两种数据：**会话**和**消息**。

- **会话（conversation）**：一次独立的对话。可以类比微信里的一个聊天窗口，有自己的标题，例如“复习SQLite外键”。一个会话包含多条消息；
- **消息（message）**：对话中的一句话。每条消息**必须属于某个会话**，不能独立存在；
- **角色（role）**：这条消息是谁说的。本课只有两种角色：`user`（用户）和`assistant`（AI助手）。

```text
会话：id=1  标题“复习SQLite外键”
  ├─ 消息1：role=user       “外键是什么？”
  ├─ 消息2：role=assistant  “外键是引用另一张表主键的列……”
  └─ 消息3：role=user       “举一个例子”

会话：id=2  标题“学习FastAPI”
  └─ 消息4：role=user       “TestClient是什么？”
```

`conversation`、`message`、`role`都不是Python关键字，是开发者定义的业务概念。在Python代码里，它们体现为Pydantic模型类名和数据库列名。

本课的角色只有`user`和`assistant`两种，且**由服务器生成**：客户端发送消息时不需要也不能提交`role`——服务器知道消息来自用户，回复当然是助手。


## 5. 数据契约：谁提交、谁生成

上一课讲过，写API前要先分清楚：哪些字段客户端必须提交，哪些可以省略，哪些由服务器生成。聊天后端的字段分成四类：

1. **客户端必须提交**（请求体字段）：例如消息正文`content`；
2. **客户端可选提交**（请求体字段，有默认值）：例如会话标题`title`，省略时服务器填默认值；
3. **路径参数**（在URL里提供，不在请求体里）：例如`/conversations/{conversation_id}/messages`中的`conversation_id`；
4. **服务器生成**（客户端绝不能提交）：例如`id`、`created_at`、`role`。

```text
请求体（客户端提交）            响应体（服务器返回）
{"content": "外键是什么"}   →   {"id": 2,
                              "conversation_id": 1,
                              "role": "assistant",
                              "content": "外键是……",
                              "created_at": "2026-08-03T14:30:00+00:00"}
```

字段说明：

- `id`：会话或消息的主键，由SQLite生成（`INTEGER PRIMARY KEY`自动递增），客户端不提交；
- `title`：会话标题，字符串；客户端可提交，省略时Pydantic填默认值；
- `content`：消息正文，字符串；客户端必须提交；
- `role`：消息角色，只能取`user`或`assistant`；由服务器生成；
- `created_at`：创建时间，服务器用标准库`datetime`生成ISO格式字符串；
- `conversation_id`：消息所属会话的id，是URL路径参数，不是请求体字段。

这些字段都不是Python关键字，是本课定义的业务字段。


### 课堂小练习1：判断字段来源

**练习目标**：巩固“谁提交、谁生成、谁在路径里”的区分。

**题目**：把下面每个字段归类，填入下列值之一：

- `"client_required_body"`：客户端必须提交的请求体字段；
- `"client_optional_body"`：客户端可选提交，省略时有默认值；
- `"client_path"`：客户端在URL路径中提供；
- `"server_generated"`：服务器生成，客户端不能提交。

先运行你自己的答案单元格，再运行紧下方的自动检测单元格。检测只覆盖代码部分，不检查解释文字。


In [3]:
# 课堂小练习1：请把 "???" 替换为上面四种值之一
field_sources = {
    "content": "client_required_body",          # 消息正文
    "conversation_id": "client_path",  # 消息所属会话
    "created_at": "server_generated",       # 创建时间
    "id": "server_generated",               # 会话/消息的主键
    "role": "server_generated",             # 消息角色
    "title": "client_optional_body",            # 会话标题
}


In [4]:
# 课堂小练习1自动检测：先运行上面的答案单元格，再运行本格。
_correct_sources = {
    "content": "client_required_body",
    "conversation_id": "client_path",
    "created_at": "server_generated",
    "id": "server_generated",
    "role": "server_generated",
    "title": "client_optional_body",
}
_check_passed = True
try:
    for _check_field, _check_expected in _correct_sources.items():
        _check_got = field_sources.get(_check_field)
        if _check_got != _check_expected:
            _check_passed = False
            print(f"❌ {_check_field}：你的答案是 {_check_got!r}，应为 {_check_expected!r}")
    if _check_passed:
        print("✅ 全部正确：字段来源判断通过。")
except NameError:
    print("❌ 还没有找到变量 field_sources，请先运行上面的答案单元格。")
except Exception as _check_error:
    print("❌ 检测时出错：", _check_error)


✅ 全部正确：字段来源判断通过。


<details><summary>完成后查看参考答案</summary>

```python
field_sources = {
    "content": "client_required_body",
    "conversation_id": "client_path",
    "created_at": "server_generated",
    "id": "server_generated",
    "role": "server_generated",
    "title": "client_optional_body",
}
```

- `content`：发送消息时客户端必须提交；
- `conversation_id`：在URL路径`/conversations/{conversation_id}/messages`中提供，不是请求体字段；
- `created_at`：服务器用`datetime`生成；
- `id`：SQLite主键自动生成；
- `role`：服务器根据消息来源生成；
- `title`：客户端可提交，省略时Pydantic填默认值。

</details>


## 6. 数据库设计：两张表和外键

第04课的示例使用内存列表保存任务，第05课才开始使用真正的SQLite表。聊天后端为什么需要**两张表**？

如果把会话和消息塞进同一张表，会出现大量重复：

```text
| id | title            | role      | content          |
|----|------------------|-----------|------------------|
| 1  | 复习SQLite       | user      | 外键是什么？     |
| 2  | 复习SQLite       | assistant | 外键是……        |
| 3  | 复习SQLite       | user      | 举一个例子       |
| 4  | 学习FastAPI      | user      | TestClient是什么？|
```

标题“复习SQLite”重复了3次；如果想改标题，要改多行；会话和消息混在一起也无法判断“哪些消息属于哪个会话”。解决办法是拆成两张表，消息表用一个**外键（foreign key）**指向会话表的主键：

- **主键（primary key）**：每张表中能唯一识别一行的列，本课是`id`；
- **外键（foreign key）**：一张表中引用另一张表主键的列。`messages.conversation_id`引用`conversations.id`，表示“这条消息属于那个会话”。

```text
conversations 表                      messages 表
| id | title     | created_at |      | id | conversation_id | role | content | created_at |
|----|-----------|------------|      |----|-----------------|------|---------|------------|
| 1  | 复习SQLite | ...        |      | 1  | 1               | user | 外键是什么？ | ... |
| 2  | 学习FastAPI| ...        |      | 2  | 1               | assistant | ... | ... |
                                      | 3  | 1               | user | 举一个例子 | ... |
                                      | 4  | 2               | user | ... | ... |
```

“一张表保存同一类对象”来自第05课；外键让“一对多”关系（一个会话对应多条消息）由数据库结构表达出来，而不是靠人的自觉。


### 外键演示（示例业务：学习卡片）

为了只看外键本身，先忽略聊天，用“学习卡片”演示：一张卡片（card）必须属于一个主题（topic）。示例业务与作业业务不同，请把注意力放在结构上。

下面的代码会：

1. 创建只在内存中的课堂数据库；
2. 建`topics`和`cards`两张表，`cards.topic_id`引用`topics.id`；
3. 用`PRAGMA foreign_key_list(cards)`观察外键配置；
4. 插入正常卡片，再尝试插入“没有主题的孤儿卡片”，观察约束生效。

这里`sqlite3`是Python标准库，`connect(":memory:")`创建内存数据库；`execute(...)`执行一条SQL；`PRAGMA foreign_key_list(cards)`返回cards表的外键信息列表，每行包含引用表、来源列和引用列等。`sqlite3.IntegrityError`是数据库违反约束时抛出的异常类型。

注意：SQLite默认**不强制**外键约束，需要先执行`PRAGMA foreign_keys = ON`。这一步只在当前连接内生效。


In [5]:
import sqlite3

# 创建只存在于当前内存中的课堂数据库。
demo_connection = sqlite3.connect(":memory:")
demo_connection.row_factory = sqlite3.Row
# 开启外键强制；不加这一句，SQLite只记录外键但不检查。
demo_connection.execute("PRAGMA foreign_keys = ON")

demo_connection.execute("""
    CREATE TABLE topics(
        id INTEGER PRIMARY KEY,
        name TEXT NOT NULL
    )
""")
demo_connection.execute("""
    CREATE TABLE cards(
        id INTEGER PRIMARY KEY,
        topic_id INTEGER NOT NULL REFERENCES topics(id),
        front TEXT NOT NULL,
        back TEXT NOT NULL
    )
""")

print("cards表的外键信息：", demo_connection.execute("PRAGMA foreign_key_list(cards)").fetchall())

# 正常插入：主题存在，卡片可以插入。
demo_connection.execute("INSERT INTO topics(name) VALUES('SQLite')")
demo_connection.execute(
    "INSERT INTO cards(topic_id, front, back) VALUES(?, ?, ?)",
    (1, "外键是什么", "引用另一张表主键的列"),
)
print("卡片数量：", demo_connection.execute("SELECT COUNT(*) FROM cards").fetchone()[0])

# 错误插入：主题999不存在，外键约束拒绝这条“孤儿卡片”。
try:
    demo_connection.execute(
        "INSERT INTO cards(topic_id, front, back) VALUES(?, ?, ?)",
        (999, "没有主题", "应该失败"),
    )
except sqlite3.IntegrityError as error:
    # 失败的写操作后回滚事务，让连接恢复到可继续使用的状态。
    demo_connection.rollback()
    print("插入不存在主题的卡片被拒绝：", error)
finally:
    # with connection 只管理事务，不会自动关闭连接，所以这里明确关闭。
    demo_connection.close()


cards表的外键信息： [<sqlite3.Row object at 0x13d72a770>]
卡片数量： 1
插入不存在主题的卡片被拒绝： FOREIGN KEY constraint failed


### 课堂小练习2：编写带外键的建表SQL

**练习目标**：动手写一条包含外键的`CREATE TABLE`语句。

**题目**：为“学习卡片”补全`cards`表的建表SQL字符串（**只写SQL，不要执行**）。要求：

- 列`id`是主键；
- 列`topic_id`是整数，不允许为空，并且引用`topics(id)`；
- 列`front`和`back`是文本，不允许为空。

先运行你的答案单元格（它只定义字符串，不会真的建表），再运行自动检测单元格。检测会真正执行你的SQL来验证外键是否写对。


In [6]:
# 课堂小练习2：请补全下面的SQL字符串（不要执行，先写好）
create_cards_sql = """
    CREATE TABLE cards(
        id INTEGER PRIMARY KEY,
        topic_id INTEGER NOT NULL REFERENCES topics(id),
        front TEXT NOT NULL,
        back TEXT NOT NULL
    )
"""


In [6]:
# 课堂小练习2自动检测：先运行上面的答案单元格，再运行本格。
try:
    _check_sql = create_cards_sql.strip()
    if not _check_sql.upper().startswith("CREATE TABLE"):
        print("❌ 你的SQL必须以 CREATE TABLE 开头。")
    else:
        _check_conn = sqlite3.connect(":memory:")
        _check_conn.row_factory = sqlite3.Row
        _check_conn.execute("CREATE TABLE topics(id INTEGER PRIMARY KEY, name TEXT NOT NULL)")
        try:
            _check_conn.execute(_check_sql)
        except sqlite3.OperationalError as _check_error:
            print("❌ 执行你的建表SQL失败：", _check_error)
        else:
            _check_fks = _check_conn.execute("PRAGMA foreign_key_list(cards)").fetchall()
            _check_fk_ok = any(
                row["table"] == "topics"
                and row["from"] == "topic_id"
                and row["to"] == "id"
                for row in _check_fks
            )
            _check_cols = {
                row["name"]: row
                for row in _check_conn.execute("PRAGMA table_info(cards)").fetchall()
            }
            _check_missing = [name for name in ("id", "topic_id", "front", "back")
                              if name not in _check_cols]
            if _check_missing:
                print("❌ cards表缺少列：", _check_missing)
            elif _check_cols["id"]["pk"] != 1:
                print("❌ id 必须是主键。")
            elif any(_check_cols[name]["notnull"] != 1
                     for name in ("topic_id", "front", "back")):
                print("❌ topic_id、front、back 都必须写 NOT NULL。")
            elif not _check_fk_ok:
                print("❌ topic_id 没有正确引用 topics(id)。")
            else:
                print("✅ 建表SQL正确：列、主键、非空约束和外键均符合要求。")
        finally:
            _check_conn.close()
except NameError:
    print("❌ 还没有找到变量 create_cards_sql，请先运行上面的答案单元格。")
except Exception as _check_error:
    print("❌ 检测时出错：", _check_error)


✅ 建表SQL正确：列、主键、非空约束和外键均符合要求。


<details><summary>完成后查看参考答案</summary>

```python
create_cards_sql = """
CREATE TABLE cards(
    id INTEGER PRIMARY KEY,
    topic_id INTEGER NOT NULL REFERENCES topics(id),
    front TEXT NOT NULL,
    back TEXT NOT NULL
)
"""
```

要点：

- `REFERENCES topics(id)`写在`topic_id`列上，表示外键指向`topics`表的`id`列；
- `NOT NULL`保证每条卡片都必须有主题；
- `topics`是父表、`cards`是子表。SQLite创建子表时不一定立即检查父表是否存在，但在插入或更新数据时会检查关系，因此实际项目仍应先创建父表，再创建子表。

</details>


## 7. repository：数据访问层接入配置

第05课你已经写过repository：函数接收`db_path`，打开连接、执行SQL、提交/回滚、关闭连接。本课的repository把路径改为**模块级配置**，原因是：repository要同时服务于路由（手动启动时用默认文件）和测试（每个测试用独立临时文件），把路径集中在一处配置，测试就能随时切换。

```text
第05课：每个函数都接收 db_path 参数
第06课：configure_database(path) 设置一次，之后所有函数使用同一个路径
```

五个已经提供或需要补全的辅助部分：

- `configure_database(db_path)`：把模块级变量`_DB_PATH`设为新路径，它只是“改设置”，不打开任何连接；
- `_get_db_path()`：读取`_DB_PATH`；还没配置时抛出`RuntimeError`，防止忘记配置；
- `_open_connection()`：**本课要完成的新连接工厂**。它每次打开一个新连接，设置`row_factory = sqlite3.Row`，并执行`PRAGMA foreign_keys = ON`；
- `_utc_now_iso()`：使用`datetime.now(timezone.utc).isoformat()`生成带时区的UTC时间，骨架已经提供；
- `_row_to_conversation(row)` / `_row_to_message(row)`：把`sqlite3.Row`转成普通业务字典。

为什么必须集中打开连接？SQLite的`PRAGMA foreign_keys = ON`只影响**当前连接**。如果每个函数各自用`sqlite3.connect()`却忘记开启外键，建表SQL虽然写了`REFERENCES`，孤儿消息仍可能被写入。作业因此要求所有数据库函数都通过`_open_connection()`取连接。

本课不要给`sqlite3.connect()`传`autocommit=False`：某些Python版本会立刻进入事务，此时再执行`PRAGMA foreign_keys = ON`不会生效。沿用默认事务行为，在任何写操作后明确`commit()`或`rollback()`即可。

另一个接口改进是：`create_conversation()`和`create_message()`都直接返回**完整字典**，而不是只返回id。这样services层可以立刻构造响应模型，不需要再查询一遍数据库。


### 最小示例：连接工厂为什么有用

下面仍以“学习卡片”为例，观察两个关键配置。它是可以直接运行的完整示例，但不包含聊天作业的函数答案。

- `sqlite3.connect(":memory:")`的第一个位置参数是数据库路径；特殊值`:memory:`表示只存在于内存；返回`sqlite3.Connection`对象；
- `row_factory`不是函数，而是连接对象的属性；设置为`sqlite3.Row`后，查询行既能写`row[0]`，也能写`row["name"]`；
- `execute("PRAGMA foreign_keys = ON")`在这个连接上开启外键检查；`execute`返回`sqlite3.Cursor`；
- `close()`关闭连接。`with connection:`只会提交或回滚事务，不会替你关闭连接。

直接运行下一格。预期外键开关是`1`，并分别看到按下标和按列名读取的结果。


In [7]:
import sqlite3

connection = sqlite3.connect(":memory:")
connection.row_factory = sqlite3.Row
connection.execute("PRAGMA foreign_keys = ON")
print("外键开关：", connection.execute("PRAGMA foreign_keys").fetchone()[0])
connection.execute("CREATE TABLE topics(id INTEGER PRIMARY KEY, name TEXT NOT NULL)")
connection.execute("INSERT INTO topics(name) VALUES(?)", ("Python",))
row = connection.execute("SELECT id, name FROM topics").fetchone()
print("按下标读取：", row[0], "；按列名读取：", row["name"])
connection.close()


外键开关： 1
按下标读取： 1 ；按列名读取： Python


### 课堂小练习3：编写学习数据库连接函数

**练习目标**：亲手完成“打开连接 → 配置行对象 → 开启外键 → 返回连接”的流程。

**题目**：补全`open_learning_connection()`。它不接收参数，必须创建内存数据库连接、设置`sqlite3.Row`、开启外键并返回连接。

完成后运行函数得到`learning_connection`。预期自动检测显示：它是连接对象、支持按列名读取，并且外键开关为1。

限制：只使用上一小节刚讲过的`sqlite3.connect`、`row_factory`、`execute`和`return`。先运行答案单元格，再运行自动检测。


In [8]:
import sqlite3
# 课堂小练习3：请补全函数中的TODO
def open_learning_connection():
    # TODO：创建内存连接，配置sqlite3.Row，开启外键，然后返回连接
    connection = sqlite3.connect(':memory:')
    connection.row_factory = sqlite3.Row
    connection.execute('PRAGMA foreign_keys = ON')
    return connection


learning_connection = open_learning_connection()


In [9]:
# 课堂小练习3自动检测：先运行上面的答案单元格，再运行本格。
try:
    if not isinstance(learning_connection, sqlite3.Connection):
        print("❌ 函数必须返回 sqlite3.Connection 对象。")
    elif learning_connection.row_factory is not sqlite3.Row:
        print("❌ 还没有把 row_factory 设置为 sqlite3.Row。")
    elif learning_connection.execute("PRAGMA foreign_keys").fetchone()[0] != 1:
        print("❌ 当前连接还没有开启外键检查。")
    else:
        learning_connection.execute("CREATE TABLE check_items(id INTEGER PRIMARY KEY, name TEXT)")
        learning_connection.execute("INSERT INTO check_items(name) VALUES(?)", ("通过",))
        _check_row = learning_connection.execute("SELECT id, name FROM check_items").fetchone()
        if _check_row["name"] != "通过":
            print("❌ 查询结果还不能按列名读取。")
        else:
            print("✅ 连接函数正确：连接类型、Row和外键开关都通过。")
except NameError:
    print("❌ 没找到 learning_connection，请先运行上面的答案单元格。")
except AttributeError as _check_error:
    print("❌ 函数可能没有返回连接对象：", _check_error)
except Exception as _check_error:
    print("❌ 检测时出错：", _check_error)
finally:
    if isinstance(globals().get("learning_connection"), sqlite3.Connection):
        learning_connection.close()


✅ 连接函数正确：连接类型、Row和外键开关都通过。


<details><summary>完成后查看参考答案</summary>

```python
def open_learning_connection():
    connection = sqlite3.connect(":memory:")
    connection.row_factory = sqlite3.Row
    connection.execute("PRAGMA foreign_keys = ON")
    return connection

learning_connection = open_learning_connection()
```

注意顺序：先打开连接，再在事务开始前开启外键。作业的`_open_connection()`只需把`:memory:`换成`_get_db_path()`取得的真实路径；不要把这个参考答案整段直接粘贴到作业。

</details>


## 8. services：业务逻辑层

repository只管“数据怎么存取”，但“发一条消息应该发生什么”属于**业务逻辑**，放在services层：

```text
发送消息的业务流程（作业实现顺序，也是验收契约）：
1. 检查会话是否存在（不存在 → 抛出 ConversationNotFoundError）
2. 把用户消息保存为 role="user" 的消息
3. 调用 generate_reply() 生成AI回复
4. 把AI回复保存为 role="assistant" 的消息
5. 把AI回复消息返回给路由层
```

顺序为什么重要？

- 先检查会话：消息不能发给不存在的会话（数据库外键也要求`conversation_id`有效）；
- 先保存用户消息再生成回复：真实AI需要完整历史，而且“用户消息先落库”保证即使回复失败，用户的话也不会丢；
- 先保存回复再返回：返回的消息必须已经在数据库里，否则客户端看到回复、但一刷新就没了。

本课的`repository.create_message(...)`返回完整字典，例如`{"id": 2, "conversation_id": 1, ...}`。因此services最后只需把这个字典交给`MessageResponse.model_validate(...)`，不需要根据id再次查询。`model_validate`是Pydantic模型类提供的方法：位置参数是要验证的字典，成功时返回`MessageResponse`实例，字段缺失或类型错误时抛出`ValidationError`。

下面用内存对象演示“保存→生成→保存→返回”的节奏。`datetime.now(timezone.utc)`是标准库`datetime`中获取当前UTC时间的写法，`isoformat()`把它转成可读的ISO格式字符串，例如`2026-08-03T14:30:00+00:00`，用作`created_at`。


In [10]:
from datetime import datetime, timezone

def current_time_iso() -> str:
    """返回当前UTC时间的ISO格式字符串。"""
    return datetime.now(timezone.utc).isoformat()

# 只用于课堂演示的内存消息仓库。
class MemoryMessageStore:
    def __init__(self):
        self._messages = []
        self._next_id = 1

    def save(self, role, content):
        message = {
            "id": self._next_id,
            "role": role,
            "content": content,
            "created_at": current_time_iso(),
        }
        self._next_id += 1
        self._messages.append(message)
        return message

    def all(self):
        return list(self._messages)


store = MemoryMessageStore()

# 业务顺序：保存用户消息 → 生成回复 → 保存回复
user_text = "外键是什么"
store.save("user", user_text)
reply_text = f"收到你的消息：{user_text}"  # 课堂演示用极简回显，作业另有规则
store.save("assistant", reply_text)

for message in store.all():
    print(message["id"], message["role"], "→", message["content"])


1 user → 外键是什么
2 assistant → 收到你的消息：外键是什么


### 课堂小练习4：发送消息步骤排序

**练习目标**：确认你理解services层“发送消息”的正确顺序。

**题目**：把下面五个步骤按正确顺序排列。用字母填`correct_order`列表。

- `a`：把AI回复保存为assistant消息
- `b`：调用generate_reply生成AI回复
- `c`：把用户消息保存为user消息
- `d`：检查会话是否存在
- `e`：把AI回复作为响应返回

先运行答案单元格，再运行自动检测。


In [11]:
# 课堂小练习4：把字母按正确顺序填入列表
correct_order = ["d", "c", "b", "a", "e"]


In [12]:
# 课堂小练习4自动检测：先运行上面的答案单元格，再运行本格。
_check_expected_order = ["d", "c", "b", "a", "e"]
try:
    if list(correct_order) == _check_expected_order:
        print("✅ 顺序正确：检查会话 → 保存用户消息 → 生成回复 → 保存回复 → 返回。")
    else:
        print("❌ 顺序不对：你填的是", correct_order)
        print("  参考流程：先检查会话，再保存用户消息，然后生成回复，再保存回复，最后返回。")
except NameError:
    print("❌ 还没有找到变量 correct_order，请先运行上面的答案单元格。")
except Exception as _check_error:
    print("❌ 检测时出错：", _check_error)


✅ 顺序正确：检查会话 → 保存用户消息 → 生成回复 → 保存回复 → 返回。


<details><summary>完成后查看参考答案</summary>

```python
correct_order = ["d", "c", "b", "a", "e"]
```

即：检查会话存在 → 保存用户消息 → 调用`generate_reply` → 保存AI回复 → 返回AI回复。如果先返回再保存，客户端看到的回复并没有落库；如果先调用回复再保存用户消息，用户消息就丢了。

</details>


## 9. ai_provider：把AI回复抽象成函数

“生成AI回复”放在`ai_provider.py`的`generate_reply(content)`函数里。为什么不直接写在services里？因为**它将来会被替换成真实的大模型API调用**：

```text
本课：generate_reply("外键是什么") → "收到你的消息：外键是什么"
未来：generate_reply(...)        → 调用大模型API，返回真正的回答
```

把“回复生成”独立成一个函数后，未来替换时只改这一个函数，services和路由不用动。这就是第04课“职责分离”在AI场景下的应用。

### 为什么测试用模拟回复，而不是真实大模型API？

1. **不需要密钥**：真实API要付费账号和密钥，作业不能要求你配置；
2. **不依赖网络**：测试必须在任何环境稳定运行，网络抖动会让测试失败；
3. **结果可预测**：模拟回复是确定的规则，测试才能断言“回复包含用户消息”；
4. **没有费用**：测试跑几十次，不能真的去调用付费API。

真实API调用会在“大模型应用开发”阶段学习；本课只要求你实现一个**规则回复**：根据用户消息内容返回非空字符串，并且让回复与输入相关（最简单是回显原文），这样测试可以验收。下面演示一种“规则回复”的写法（示例业务：待办助手；作业的回复规则由你自己设计，只需要满足“非空、与输入相关”）。


In [13]:
# 示例：规则回复（待办助手业务，仅演示“分支规则”的写法）
def assistant_reply_for_todo(content: str) -> str:
    """待办助手风格的模拟回复：问候走问候分支，其他内容走记录分支。"""
    if "你好" in content:
        return f"你好！我是待办助手。你说：{content}"
    return f"好的，已记录：{content}"


print(assistant_reply_for_todo("你好，帮我记一下买牛奶"))
print(assistant_reply_for_todo("今天要复习外键"))


你好！我是待办助手。你说：你好，帮我记一下买牛奶
好的，已记录：今天要复习外键


### 课堂小练习5：判断说法

**练习目标**：确认你理解“模拟回复 vs 真实API”和“为什么独立成函数”。

**题目**：判断下面每个说法的对错，填`True`或`False`。

先运行答案单元格，再运行自动检测。


In [14]:
# 课堂小练习5：把 "???" 替换为 True 或 False
provider_judgments = {
    "真实大模型API需要网络和密钥，测试用它结果不稳定": True,
    "模拟回复让测试不依赖外部服务就能验收业务流程": True,
    "回复生成应该直接写在路由函数里，这样代码最短": False,
    "把回复生成独立成函数后，未来换成真实API只改一个地方": True,
}


In [15]:
# 课堂小练习5自动检测：先运行上面的答案单元格，再运行本格。
_correct_judgments = {
    "真实大模型API需要网络和密钥，测试用它结果不稳定": True,
    "模拟回复让测试不依赖外部服务就能验收业务流程": True,
    "回复生成应该直接写在路由函数里，这样代码最短": False,
    "把回复生成独立成函数后，未来换成真实API只改一个地方": True,
}
_check_passed = True
try:
    for _check_statement, _check_expected in _correct_judgments.items():
        _check_got = provider_judgments.get(_check_statement)
        if _check_got is not _check_expected:
            _check_passed = False
            print(f"❌ “{_check_statement}” 你的答案是 {_check_got!r}，应为 {_check_expected!r}")
    if _check_passed:
        print("✅ 全部正确：provider概念判断题通过。")
except NameError:
    print("❌ 还没有找到变量 provider_judgments，请先运行上面的答案单元格。")
except Exception as _check_error:
    print("❌ 检测时出错：", _check_error)


✅ 全部正确：provider概念判断题通过。


<details><summary>完成后查看参考答案</summary>

```python
provider_judgments = {
    "真实大模型API需要网络和密钥，测试用它结果不稳定": True,
    "模拟回复让测试不依赖外部服务就能验收业务流程": True,
    "回复生成应该直接写在路由函数里，这样代码最短": False,
    "把回复生成独立成函数后，未来换成真实API只改一个地方": True,
}
```

第三条是错的：路由函数只负责HTTP分发；把回复生成写进路由会让路由既管HTTP又管业务，违反第04课的分层原则，也难测试。

</details>


## 10. routers：HTTP路由和404

路由层沿用第04课的`APIRouter`写法：给一组路由配置共同前缀和标签。本课新增一个概念：**`HTTPException`**。

### 会话不存在时返回404

services层遇到不存在的会话时抛出`ConversationNotFoundError`。路由函数捕获它，并抛出`HTTPException`，让FastAPI返回404响应。

- `HTTPException`来自`fastapi`模块，是FastAPI提供的“带状态码的HTTP错误”类；
- `HTTPException(status_code=404, detail="会话不存在")`中：
  - `status_code`：HTTP状态码，404表示“找不到资源”；
  - `detail`：错误说明，会出现在响应体的`"detail"`字段里；
- 抛出`HTTPException`后，**用户代码里没有“把异常转成响应”的那一行**：FastAPI框架会自动拦截它，构造出`{"detail": "会话不存在"}`响应并返回给客户端；
- 对比：普通Python异常会一路冒泡，最终变成500内部错误；`HTTPException`是“有意的业务错误”，FastAPI按它的状态码生成响应。

下面用“学习主题”演示同样的模式（示例业务：`/topics/{topic_id}`，作业是`/conversations/{conversation_id}`）。


In [16]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient

demo_app = FastAPI()


@demo_app.get("/topics/{topic_id}")
def get_topic(topic_id: int):
    """只认识id为1的主题；其他id一律返回404。"""
    if topic_id == 1:
        return {"id": 1, "name": "SQLite"}
    raise HTTPException(status_code=404, detail="主题不存在")


demo_client = TestClient(demo_app, raise_server_exceptions=False)

exists = demo_client.get("/topics/1")
print("存在的主题：", exists.status_code, exists.json())

missing = demo_client.get("/topics/999")
print("不存在的主题：", missing.status_code, missing.json())


存在的主题： 200 {'id': 1, 'name': 'SQLite'}
不存在的主题： 404 {'detail': '主题不存在'}


上面的输出应该类似：

```text
存在的主题： 200 {'id': 1, 'name': 'SQLite'}
不存在的主题： 404 {'detail': '主题不存在'}
```

注意：`raise_server_exceptions=False`让`TestClient`把服务器内部错误转成500响应，避免未完成代码打印超长错误栈；它不改变`HTTPException(404)`的正常行为。

### 最终路径怎么计算

第04课讲过：`router = APIRouter(prefix="/conversations")`，路由装饰器里的路径会拼在前缀后面。本课作业的5个接口：

| 装饰器路径 | 最终路径 |
|---|---|
| `@router.post("")` | `POST /conversations` |
| `@router.get("")` | `GET /conversations` |
| `@router.get("/{conversation_id}")` | `GET /conversations/{conversation_id}` |
| `@router.post("/{conversation_id}/messages")` | `POST /conversations/{conversation_id}/messages` |
| `@router.get("/{conversation_id}/messages")` | `GET /conversations/{conversation_id}/messages` |

`POST`和`GET`可以用同一个路径：HTTP方法不同，路由就不同。装饰器里的`{conversation_id}`是**路径参数**，FastAPI会把URL中的数字解析出来传给函数参数`conversation_id`。


### 课堂小练习6：计算最终路径

**练习目标**：确认你会计算“前缀 + 装饰器路径 = 最终路径”。

**题目**：`router`前缀是`/conversations`，请写出每个接口的最终完整路径（例如`/conversations`或`/conversations/{conversation_id}`）。

先运行答案单元格，再运行自动检测。


In [19]:
# 课堂小练习6：把 "???" 替换为最终完整路径
final_paths = {
    "POST ''": "/conversations",
    "GET ''": "/conversations",
    "GET '/{conversation_id}'": "/conversations/{conversation_id}",
    "POST '/{conversation_id}/messages'": "/conversations/{conversation_id}/messages",
    "GET '/{conversation_id}/messages'": "/conversations/{conversation_id}/messages",
}


In [20]:
# 课堂小练习6自动检测：先运行上面的答案单元格，再运行本格。
_correct_paths = {
    "POST ''": "/conversations",
    "GET ''": "/conversations",
    "GET '/{conversation_id}'": "/conversations/{conversation_id}",
    "POST '/{conversation_id}/messages'": "/conversations/{conversation_id}/messages",
    "GET '/{conversation_id}/messages'": "/conversations/{conversation_id}/messages",
}
_check_passed = True
try:
    for _check_key, _check_expected in _correct_paths.items():
        _check_got = final_paths.get(_check_key)
        if _check_got != _check_expected:
            _check_passed = False
            print(f"❌ {_check_key}：你的答案是 {_check_got!r}，应为 {_check_expected!r}")
    if _check_passed:
        print("✅ 全部正确：最终路径计算通过。")
except NameError:
    print("❌ 还没有找到变量 final_paths，请先运行上面的答案单元格。")
except Exception as _check_error:
    print("❌ 检测时出错：", _check_error)


✅ 全部正确：最终路径计算通过。


<details><summary>完成后查看参考答案</summary>

```python
final_paths = {
    "POST ''": "/conversations",
    "GET ''": "/conversations",
    "GET '/{conversation_id}'": "/conversations/{conversation_id}",
    "POST '/{conversation_id}/messages'": "/conversations/{conversation_id}/messages",
    "GET '/{conversation_id}/messages'": "/conversations/{conversation_id}/messages",
}
```

`""`（空字符串）拼在前缀后面仍是`/conversations`；`/conversations` 可以同时挂`GET`和`POST`，因为方法不同。

</details>


## 11. main：组装应用与lifespan

`main.py`负责三件事：

1. 创建应用`app = FastAPI(...)`；
2. 把聊天路由组挂载进去`app.include_router(chat_router)`；
3. **在应用启动时初始化数据库**。

第3件事用FastAPI的**`lifespan`（生命周期）**机制实现。它是一个异步上下文管理器：

- `asynccontextmanager`来自标准库`contextlib`，它是函数；`@asynccontextmanager`把下面这个含`yield`的异步生成器函数包装成FastAPI能使用的异步上下文管理器；
- `lifespan(app)`中的`app`由FastAPI传入，类型是`FastAPI`；本课启动逻辑不需要读取它，但函数签名仍要保留；
- 返回类型`AsyncIterator[None]`表示函数运行到`yield`时把控制权交给应用，不额外产生业务数据；
- `yield`之前的代码：在**应用启动时**执行（uvicorn启动应用时自动运行）；
- `yield`之后的代码：在**应用关闭时**执行（清理资源，本课不需要）；
- 这样“先建表、再对外服务”的顺序由框架保证，不需要在路由里每次检查表是否存在。

```python
@asynccontextmanager
async def demo_lifespan(app: FastAPI) -> AsyncIterator[None]:
    # 示例业务：启动时准备学习记录数据库；作业要迁移这个思路。
    learning_path = Path(__file__).resolve().parent / "learning.sqlite3"
    print("启动时准备：", learning_path)
    yield
```

### TestClient什么时候触发lifespan？

- `with TestClient(app) as client:` 会触发启动和关闭代码；
- `client = TestClient(app)`（不带`with`，本课测试文件的做法）**不会触发**；
- 因此本课分层测试通过辅助函数`_prepare_database(tmp_path)`自己配置临时数据库，不依赖lifespan，也不会碰到真实数据文件。`tmp_path`是pytest提供的临时目录，每个测试拿到不同路径。

下面的小演示展示异步上下文管理器“进入时执行、退出时执行”的节奏（与FastAPI无关的纯Python演示）。


In [21]:
from contextlib import asynccontextmanager


@asynccontextmanager
async def demo_lifespan():
    print("启动：初始化数据库表结构")
    yield
    print("关闭：清理资源")


async def run_demo():
    async with demo_lifespan():
        print("应用运行中，可以处理请求")


# Notebook顶层支持await；如果作为普通.py脚本运行，请改用 asyncio.run(run_demo())。
await run_demo()


启动：初始化数据库表结构
应用运行中，可以处理请求
关闭：清理资源


### 手动启动作业应用

作业完成后，在本课目录执行：

```powershell
uvicorn chat_app.main:app --reload
```

- `uvicorn`：ASGI服务器；
- `chat_app.main:app`：`模块:变量`语法，表示从`chat_app`包的`main`模块导入`app`变量；
- `--reload`：代码变化时自动重启（开发用）。

启动后打开 <http://127.0.0.1:8000/docs> 可以看到Swagger自动文档，可以直接在页面上试接口；`http://127.0.0.1:8000/health` 返回`{"status": "ok"}`。按`Ctrl+C`停止服务器。

如果启动时报`NotImplementedError`，说明作业还没完成，lifespan里的数据库初始化还没有实现。


## 12. 完整执行流程回顾

把每一层串起来看一次`POST /conversations/1/messages`请求（假设会话1已存在）：

```text
1. 客户端发送：POST /conversations/1/messages
   请求体 {"content": "外键是什么"}
2. FastAPI框架解析URL：路径参数 conversation_id=1（来自URL）
   校验请求体：按MessageCreate模型解析，content必须是非空字符串
3. 路由函数 send_message_endpoint(conversation_id, message) 被调用
4. services.send_message：
   a. repository.get_conversation(1) → 存在
   b. repository.create_message(1, "user", "外键是什么") → 保存用户消息
   c. ai_provider.generate_reply("外键是什么") → 模拟回复
   d. repository.create_message(1, "assistant", 模拟回复) → 保存AI回复
   e. create_message直接返回完整AI消息字典，services将其验证成MessageResponse
5. FastAPI框架按response_model序列化：201 + JSON响应体
6. 客户端收到：{"id": 2, "conversation_id": 1, "role": "assistant", ...}
```

哪些是框架做的：路由匹配、路径参数解析、请求体校验、响应序列化、`HTTPException`转响应。哪些是你写的：路由函数、services、repository、ai_provider。哪些是SQLite做的：建表、约束、事务、持久化文件。


## 13. 常见错误与定位顺序

按“现象 → 先查哪里 → 原因”的顺序排查：

| 现象 | 先查哪里 | 常见原因 |
|---|---|---|
| `ModuleNotFoundError: chat_app` | 执行命令的目录 | 不在本课目录运行；`uvicorn chat_app.main:app`必须在本课目录（含`chat_app`包）下执行 |
| `RuntimeError: 尚未配置数据库路径` | `main.py`的lifespan、测试的fixture | 忘记调用`repository.configure_database()` |
| `sqlite3.OperationalError: no such table` | `initialize_database` | 建表SQL还没实现，或没调用初始化 |
| 发送消息返回500 | services的`send_message` | 会话检查、保存顺序有误，或`generate_reply`返回了空字符串 |
| 消息列表是空的 | `list_messages` | 忘了`commit()`，或SQL漏了`WHERE conversation_id = ?` |
| 请求返回422 | 请求体 | `content`为空/超长，或字段名写错（注意是`content`不是`message`） |
| 更新数据后重启丢失 | 数据库路径 | `configure_database`用了`:memory:`或临时路径，真正的文件路径在`chat.sqlite3` |

定位顺序建议：先看`health`是否200（应用是否起来）→ 再看`POST /conversations`能否创建 → 再看`GET /conversations/{id}/messages`能否读出消息 → 最后看发送消息的完整流程。逐层缩小范围，不要一次改多个文件。


## 14. 课末作业：AI聊天后端项目

### 任务背景

完成`chat_app`包，实现一个可持久化的AI聊天后端：支持创建会话、查看会话、在会话中发消息，每次发送消息都会保存用户消息并自动生成、保存一条模拟AI回复。作业综合了第04课的分层结构、第05课的SQLite数据访问和本课新学的业务概念。

### 允许修改的范围

只修改以下文件中的TODO区域：

- `chat_app/schemas.py`
- `chat_app/repository.py`
- `chat_app/services.py`
- `chat_app/ai_provider.py`
- `chat_app/routers/chat.py`
- `chat_app/main.py`

不要修改`test_chat_app.py`来降低要求，也不需要修改两个`__init__.py`。题目需要的导入已经全部放在各文件顶部；你不需要在TODO区域外补导入。`repository.py`中的路径、时间和行转换辅助函数已经提供；`_open_connection()`本身是TODO。

**建议实现顺序（不是答案）：**先`schemas.py`和`ai_provider.py`；再完成`repository.py`中的连接、建表、会话函数、消息函数；然后完成`services.py`、`routers/chat.py`和`main.py`。下面提供每一层独立的测试命令，前一层未完成不会阻止你检查schema或provider。

### 表结构契约

数据库文件默认是`chat.sqlite3`，由`main.py`的lifespan创建。

`conversations`表：

| 列 | SQLite类型 | 约束和默认值 | 数据由谁提供 |
|---|---|---|---|
| `id` | `INTEGER` | `PRIMARY KEY` | SQLite生成，调用者不提交 |
| `title` | `TEXT` | `NOT NULL`，长度1至50 | 调用者提供（Pydantic已校验） |
| `created_at` | `TEXT` | `NOT NULL` | 服务器生成，ISO格式字符串 |

`messages`表：

| 列 | SQLite类型 | 约束和默认值 | 数据由谁提供 |
|---|---|---|---|
| `id` | `INTEGER` | `PRIMARY KEY` | SQLite生成，调用者不提交 |
| `conversation_id` | `INTEGER` | `NOT NULL`，引用`conversations(id)` | 路径参数，services传入 |
| `role` | `TEXT` | `NOT NULL`，只能`user`或`assistant` | 服务器生成 |
| `content` | `TEXT` | `NOT NULL`，长度1至2000 | user内容来自请求；assistant内容由服务器生成 |
| `created_at` | `TEXT` | `NOT NULL` | 服务器生成，ISO格式字符串 |

要求：`initialize_database()`重复调用不得删除已有数据（用`CREATE TABLE IF NOT EXISTS`）；`messages`表必须包含指向`conversations(id)`的外键；标题、角色和内容范围都要写成数据库`CHECK`约束，不能只依赖Pydantic。

为什么请求模型允许1至500，而数据库允许1至2000？客户端输入只允许500字符，但服务器生成的AI回复可能在原消息之外增加解释文字，因此响应内容需要更大的存储上限。这两个范围不同是有意设计，不是笔误。

### 数据契约：字段到底由谁提供

`ConversationCreate`（创建会话请求模型）：

| 字段 | Python类型 | 是否必填/默认 | 约束 | 数据由谁提供 |
|---|---|---|---|---|
| `title` | `str` | 非必填、默认`"新会话"` | 长度1至50 | 客户端可提交；省略时Pydantic填默认值 |

`ConversationResponse`（会话响应模型）：

| 字段 | Python类型 | 响应值来自哪里 | 客户端创建时是否提交 |
|---|---|---|---|
| `id` | `int` | 服务器按创建顺序生成，从1开始 | 否 |
| `title` | `str` | 从已验证的请求数据取得，可能是默认值 | 可选 |
| `created_at` | `str` | 服务器用`datetime`生成 | 否 |

`MessageCreate`（发送消息请求模型）：

| 字段 | Python类型 | 是否必填/默认 | 约束 | 数据由谁提供 |
|---|---|---|---|---|
| `content` | `str` | 必填、无默认值 | 长度1至500 | 客户端必须提交 |

`MessageResponse`（消息响应模型）：

| 字段 | Python类型 | 响应值来自哪里 | 客户端发送时是否提交 |
|---|---|---|---|
| `id` | `int` | 服务器生成，消息id全局递增 | 否 |
| `conversation_id` | `int` | 从URL路径参数取得 | 否（在路径中） |
| `role` | `str` | 服务器生成，`user`或`assistant` | 否 |
| `content` | `str` | 用户消息来自请求体；AI回复由服务器生成 | 用户消息是；AI回复不是 |
| `created_at` | `str` | 服务器用`datetime`生成 | 否 |

`title`、`content`、`role`、`created_at`、`conversation_id`都不是Python关键字，是本作业定义的业务字段。客户端不要提交`id`、`role`、`created_at`；`conversation_id`通过URL路径提供，不在请求体里。


### HTTP接口契约

主应用标题必须是`AI聊天后端`。聊天路由组的共同前缀是`/conversations`，自动文档标签是`chat`。

| 方法与最终路径 | 客户端输入 | 成功状态码 | 成功响应体 |
|---|---|---|---|
| `GET /health` | 无 | 200 | `{"status": "ok"}`（已提供） |
| `POST /conversations` | `ConversationCreate` JSON请求体 | 201 | 新创建的完整`ConversationResponse` |
| `GET /conversations` | 无 | 200 | 按id升序的`ConversationResponse`列表；初始为空列表 |
| `GET /conversations/{conversation_id}` | 路径参数 | 200 | 对应`ConversationResponse`；不存在时404 `{"detail": "会话不存在"}` |
| `POST /conversations/{conversation_id}/messages` | `MessageCreate` JSON请求体 | 201 | 服务器生成的AI回复`MessageResponse` |
| `GET /conversations/{conversation_id}/messages` | 路径参数 | 200 | 按id升序的`MessageResponse`列表；没有消息时为空列表 |

错误行为：

- 验证失败由FastAPI自动返回422：缺少`content`、空`content`、`content`超过500字符、空`title`、`title`超过50字符；题目不要求自定义422文本；
- 会话不存在时，`GET /conversations/{id}`、发消息、查消息都返回404 `{"detail": "会话不存在"}`；
- 发送消息的返回值是**AI回复消息**（`role="assistant"`），用户消息不直接返回，但已经保存；通过`GET .../messages`可以看到`user`和`assistant`两条消息按保存顺序排列；
- `generate_reply`必须返回**非空字符串**，并且回复内容与用户消息相关（测试用“回复包含用户消息原文”验收，最简单实现是回显；这是验收标准，不是唯一实现方式）；
- 消息`id`全局递增，不按会话重新编号：第1次发送产生id 1（user）和2（assistant），第2次发送产生id 3和4。

### 各文件必须实现的职责（这是验收契约，不是代码答案）

- `schemas.py`：定义四个Pydantic模型及字段约束（默认值、长度范围）；
- `repository.py`：`_open_connection`统一配置`sqlite3.Row`和当前连接的外键开关；`initialize_database`建两张表；五个增查函数使用`?`占位符；写操作成功`commit()`、失败`rollback()`、所有连接最终`close()`；两个create函数返回完整字典；
- `services.py`：`create_conversation`调用repository组装响应；`list_conversations`返回全部会话；`get_conversation`按id查询、不存在时抛`ConversationNotFoundError`；`send_message`按“检查会话→保存用户消息→生成回复→保存回复→返回回复”实现；`list_messages`先检查会话再返回消息列表；会话不存在时抛`ConversationNotFoundError`；
- `ai_provider.py`：`generate_reply`实现非空、与输入相关的规则回复；
- `routers/chat.py`：配置`prefix="/conversations"`和`tags=["chat"]`；定义5个接口，成功状态码和响应模型正确；捕获`ConversationNotFoundError`转成`HTTPException(404, "会话不存在")`；路由函数只做分发，不写SQL、不写回复规则；
- `main.py`：标题`AI聊天后端`；挂载聊天路由组；lifespan中配置默认数据库路径（本课目录下的`chat.sqlite3`）并调用`repository.initialize_database()`。

### 运行作业测试

在本课目录执行：

```powershell
python -m pytest test_chat_app.py -v
```

应收集**34个测试项**；全部完成后应为`34 passed`。测试覆盖字段边界、外键是否真正启用、表约束、重复初始化保留数据、ISO时间、lifespan启动行为、分层业务错误、500字符边界、404和真实文件持久化。

推荐分层完成和运行：

```powershell
python -m pytest test_chat_app.py -v -m schema
python -m pytest test_chat_app.py -v -m provider
python -m pytest test_chat_app.py -v -m repository
python -m pytest test_chat_app.py -v -m service
python -m pytest test_chat_app.py -v -m api
```

`-m`是pytest的marker筛选参数，例如`-m schema`只运行带`@pytest.mark.schema`的测试。测试没有使用`autouse`初始化器：schema和provider可以最先独立通过。需要数据库的测试才调用`_prepare_database(tmp_path)`；`tmp_path`由pytest提供临时目录，不会把测试数据库写到固定的C盘项目目录。

`TestClient(app, raise_server_exceptions=False)`的`app`是要测试的FastAPI对象；关键字参数`raise_server_exceptions=False`把未完成代码导致的服务器异常表现成500响应，减少超长错误栈。最后一个测试会绕过API直接读取SQLite文件，确认数据确实落盘。

手动启动并体验：

```powershell
uvicorn chat_app.main:app --reload
```

然后打开 <http://127.0.0.1:8000/docs>，依次创建会话、发送消息、查看消息列表。


## 15. 理解题

请用自己的话回答（每题在Notebook前文都有讲解和示例）：

1. 为什么“会话”和“消息”要分成两张表，而不是存在同一张表里？`messages.conversation_id`外键解决什么问题？
2. 发送一条消息时，services层按照什么顺序做了哪些事？为什么“先保存用户消息”而不是“先调用`generate_reply`”？
3. 请求模型`MessageCreate`和响应模型`MessageResponse`的字段有什么不同？`role`和`created_at`为什么不由客户端提交？
4. repository、services、routers三层的职责边界分别是什么？为什么路由函数不能直接写SQL？
5. 为什么测试使用模拟回复而不是真实大模型API？把回复生成独立成`ai_provider`函数有什么好处？
6. `HTTPException(404, "会话不存在")`抛出后，谁把它变成HTTP响应？它与普通Python异常冒泡成500有什么不同？
7. `lifespan`里的代码什么时候执行？为什么测试文件（不带`with`的`TestClient`）不受lifespan影响？


### 在这里填写理解题答案

**1. 为什么要分成两张表、外键解决什么问题**

会话保存标题、创建时间等会话本身的信息，消息保存角色、内容和发送时间等消息信息。一个会话可以包含多条消息，如果放在同一张表中，会话标题等数据会被重复保存，修改标题时还要更新多行，也难以清楚表达一对多关系。`messages.conversation_id`记录每条消息属于哪个会话；外键还要求这个编号必须对应真实存在的`conversations.id`，防止产生找不到所属会话的孤立消息。

**2. 发送消息的流程与顺序**

services层先查询并确认会话存在，然后保存用户消息，再把用户内容交给`generate_reply`生成回复，接着保存assistant消息，最后把完整的assistant消息转换成响应模型并返回。先保存用户消息可以保证对话输入已经留下记录，而且让数据库中的顺序保持为user在前、assistant在后；如果先生成回复，生成过程失败或流程中断时，用户实际发送过的内容可能没有记录。

**3. 请求模型与响应模型的区别**

`MessageCreate`只包含客户端必须提交的`content`。`MessageResponse`还包含`id`、`conversation_id`、`role`和`created_at`，用于描述服务器已经保存的完整消息。`id`由数据库生成，`conversation_id`来自URL路径，`role`由服务器根据消息是用户消息还是AI回复来决定，`created_at`由服务器生成。后两者不能交给客户端填写，否则客户端可以伪装消息身份或提交不可信的时间。

**4. 三层职责边界**

repository层只负责SQLite连接、SQL执行以及把数据库行转换成数据；services层负责业务流程和规则，例如检查会话、按顺序保存两条消息、生成回复；routers层负责接收HTTP参数、调用services、声明响应模型和状态码，并把业务异常转换成HTTP错误。路由直接写SQL会把HTTP协议和数据存储耦合在一起，使代码难以复用、测试和更换数据库，也容易让一个函数承担过多职责。

**5. 模拟回复与独立函数**

测试使用模拟回复，是为了避免依赖真实密钥和网络，避免产生调用费用，并让相同输入得到可预测的结果，使测试稳定。把生成回复单独放进`ai_provider.generate_reply`后，services只依赖统一的函数接口；以后接入真实模型时主要替换provider中的实现，不必同时修改路由、数据库和业务流程，测试时也更容易替换成假的实现。

**6. HTTPException如何变成响应**

路由抛出`HTTPException(404, "会话不存在")`后，FastAPI会捕获这个框架认识的异常，根据其中的`status_code`和`detail`生成404 HTTP响应。普通Python异常没有携带预期的HTTP语义，如果一直向上冒泡且没有被处理，服务器通常会把它当成程序内部错误并返回500，而不是告诉客户端资源不存在。

**7. lifespan的执行时机**

FastAPI进入应用生命周期时先执行`yield`之前的代码，所以数据库路径配置和建表会在接收请求前完成；应用关闭时再继续执行`yield`之后的清理代码。不使用`with`上下文管理器创建的`TestClient`不会进入完整的启动和关闭生命周期，因此不会自动运行lifespan。本项目的大部分测试会自行配置和初始化临时数据库，这样测试彼此隔离，也不会写入正式的`chat.sqlite3`。


## 16. 本课小结

你已经把前五课的知识组合成了一个完整后端：

- **第02/03课**：FastAPI路由和Pydantic模型负责HTTP和校验；
- **第04课**：schemas/services/routers/main分层负责职责分离；
- **第05课**：repository + SQLite负责持久化；
- **本课新增**：两张表和外键表达“一对多”关系，services组合业务流程，`ai_provider`抽象AI回复，`lifespan`初始化数据库，`HTTPException`处理业务错误。

下一阶段（03 Docker与部署）会把这样的应用打包部署；再往后（大模型应用开发）会把`generate_reply`从规则回复替换成真实大模型API——因为你已经把它独立成一个函数，替换时只改一个文件。

## 可选参考资料

- [FastAPI官方文档：路径参数](https://fastapi.tiangolo.com/zh/tutorial/path-params/)
- [FastAPI官方文档：请求体](https://fastapi.tiangolo.com/zh/tutorial/body/)
- [FastAPI官方文档：lifespan](https://fastapi.tiangolo.com/zh/advanced/events/)
- [Python官方文档：sqlite3](https://docs.python.org/3/library/sqlite3.html)
- [SQLite官方文档：外键约束](https://www.sqlite.org/foreignkeys.html)

官方文档信息完整但默认读者已经了解相关概念；第一次学习建议先按本Notebook顺序运行，再对照文档查阅。
